# AgentCore Observability — Monitor Agent Behavior in Production

This notebook demonstrates:
- Enabling OpenTelemetry tracing for agents
- Viewing traces in CloudWatch (full agent loop visibility)
- Monitoring key metrics (token usage, latency, errors)
- Creating a CloudWatch dashboard for agent monitoring
- Debugging failed tool calls using trace data

## ⚠️ Cost Warning
- CloudWatch Logs: Minimal cost for trace data
- CloudWatch Metrics: Standard pricing per metric
- Model invocations: Standard Bedrock pricing
- Estimated cost for this lab: **< $0.50**
- **Cleanup**: Delete CloudWatch dashboard and log groups after the lab

In [1]:
# Install required packages
!pip install boto3 strands-agents strands-agents-tools opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp -q

In [2]:
import boto3
import json
import time
from datetime import datetime, timedelta
from strands import Agent, tool
from strands.models import BedrockModel

REGION = "us-west-2"
MODEL_ID = "us.amazon.nova-pro-v1:0"

cloudwatch = boto3.client("cloudwatch", region_name=REGION)
logs_client = boto3.client("logs", region_name=REGION)
model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

print(f"Region: {REGION}")
print(f"Model: {MODEL_ID}")

Region: us-west-2
Model: us.amazon.nova-pro-v1:0


## 1. Create an Agent with Instrumentation

We'll create an agent and manually instrument it to capture traces and metrics.

In [3]:
# Simple metrics collector
class AgentMetrics:
    def __init__(self):
        self.invocations = []
    
    def record(self, session_id, latency, input_tokens, output_tokens, tool_calls, success):
        self.invocations.append({
            "timestamp": datetime.utcnow().isoformat(),
            "session_id": session_id,
            "latency_ms": int(latency * 1000),
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "tool_calls": tool_calls,
            "success": success
        })
    
    def summary(self):
        if not self.invocations:
            return "No data"
        latencies = [i["latency_ms"] for i in self.invocations]
        tokens = [i["input_tokens"] + i["output_tokens"] for i in self.invocations]
        errors = sum(1 for i in self.invocations if not i["success"])
        return {
            "total_invocations": len(self.invocations),
            "avg_latency_ms": sum(latencies) // len(latencies),
            "p95_latency_ms": sorted(latencies)[int(len(latencies) * 0.95)],
            "total_tokens": sum(tokens),
            "error_count": errors,
            "error_rate": f"{errors/len(self.invocations)*100:.1f}%"
        }

metrics = AgentMetrics()
print("Metrics collector initialized")

Metrics collector initialized


In [4]:
# Tools with instrumentation
@tool
def search_docs(query: str) -> str:
    """Search documentation for an answer.
    
    Args:
        query: The search query
    """
    # Simulate search with varying latency
    time.sleep(0.1)  # Simulate API call
    docs = {
        "pricing": "Bedrock pricing is per-token for on-demand inference.",
        "limits": "Default quota is 100 requests per minute per model.",
        "regions": "Bedrock is available in us-east-1, us-west-2, eu-west-1, and more."
    }
    for key, value in docs.items():
        if key in query.lower():
            return value
    return "No relevant documentation found."

@tool
def failing_tool(input_text: str) -> str:
    """A tool that sometimes fails (for testing error observability).
    
    Args:
        input_text: Input to process
    """
    if "error" in input_text.lower():
        raise Exception("Simulated tool failure: connection timeout")
    return f"Processed: {input_text}"

# Create instrumented agent
agent = Agent(
    model=model,
    tools=[search_docs, failing_tool],
    system_prompt="You are a helpful assistant. Use search_docs to find information. Use failing_tool only when explicitly asked."
)

print("Instrumented agent created with search_docs and failing_tool")

Instrumented agent created with search_docs and failing_tool


## 2. Run Agent with Trace Collection

Execute several requests and collect trace data.

In [5]:
# Run multiple requests to generate trace data
test_prompts = [
    "What is Bedrock pricing?",
    "What are the rate limits?",
    "Which regions support Bedrock?",
    "Tell me about Lambda integration",  # Will get "no docs found"
]

traces = []
for i, prompt in enumerate(test_prompts):
    session_id = f"session-{i+1}"
    start = time.time()
    
    try:
        result = agent(prompt)
        latency = time.time() - start
        
        # Extract token usage from result metadata
        input_tokens = getattr(result, 'input_tokens', 100)  # Approximate if not available
        output_tokens = getattr(result, 'output_tokens', 50)
        
        metrics.record(session_id, latency, input_tokens, output_tokens, tool_calls=1, success=True)
        traces.append({"session": session_id, "prompt": prompt, "status": "success", "latency": latency})
        print(f"  ✅ {session_id}: {prompt[:40]}... ({latency:.2f}s)")
    except Exception as e:
        latency = time.time() - start
        metrics.record(session_id, latency, 0, 0, tool_calls=0, success=False)
        traces.append({"session": session_id, "prompt": prompt, "status": "error", "error": str(e)})
        print(f"  ❌ {session_id}: {prompt[:40]}... ERROR: {e}")

print(f"\nCollected {len(traces)} traces")

<thinking> To find information about Bedrock pricing, I need to search the documentation using the `search_docs` tool. I will use a query that directly asks for Bedrock pricing information. </thinking>

Tool #1: search_docs
<thinking> The documentation indicates that Bedrock pricing is per-token for on-demand inference. To provide a more detailed answer, I should search for specific details about the per-token pricing. </thinking> 
Tool #2: search_docs
<thinking> The documentation still only provides the general information that Bedrock pricing is per-token for on-demand inference. It does not offer specific details about the per-token pricing. Since the available tool results do not provide the detailed pricing information, I will inform the user about the general pricing model and suggest they check the official AWS pricing page for detailed information. </thinking>

Bedrock pricing is based on a per-token model for on-demand inference. For detailed pricing information, please refer 

/var/folders/jn/py_6v4j90519z5d9j8hh442r0000gn/T/ipykernel_51331/2095800752.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


<thinking> To find information about Bedrock rate limits, I need to search the documentation using the `search_docs` tool. I will use a query that directly asks for Bedrock rate limit information. </thinking> 
Tool #3: search_docs
<thinking> The documentation indicates that the default quota for Bedrock is 100 requests per minute per model. To provide a more comprehensive answer, I should search for any additional details or exceptions to this rate limit. </thinking> 
Tool #4: search_docs
<thinking> The documentation only provides the default quota of 100 requests per minute per model for Bedrock. Since the available tool results do not provide additional details or exceptions to this rate limit, I will inform the user about the default rate limit and suggest they check the official AWS documentation for any updates or exceptions. </thinking>

The default quota for Bedrock is 100 requests per minute per model. For any exceptions or updates to this rate limit, please refer to the offici

## 3. View Metrics Summary

In [6]:
summary = metrics.summary()
print("=== Agent Metrics Summary ===")
print(f"{'Metric':<25} {'Value':<15}")
print("=" * 40)
for key, value in summary.items():
    print(f"{key:<25} {value:<15}")

=== Agent Metrics Summary ===
Metric                    Value          
total_invocations         4              
avg_latency_ms            3141           
p95_latency_ms            3883           
total_tokens              600            
error_count               0              
error_rate                0.0%           


## 4. Publish Metrics to CloudWatch

Push agent metrics to CloudWatch for dashboarding and alerting.

In [7]:
# Publish custom metrics to CloudWatch
NAMESPACE = "AgentCore/Lab"

metric_data = []
for inv in metrics.invocations:
    metric_data.extend([
        {
            "MetricName": "AgentLatency",
            "Value": inv["latency_ms"],
            "Unit": "Milliseconds",
            "Dimensions": [{"Name": "AgentName", "Value": "support-agent"}]
        },
        {
            "MetricName": "TokenUsage",
            "Value": inv["input_tokens"] + inv["output_tokens"],
            "Unit": "Count",
            "Dimensions": [{"Name": "AgentName", "Value": "support-agent"}]
        },
        {
            "MetricName": "Errors",
            "Value": 0 if inv["success"] else 1,
            "Unit": "Count",
            "Dimensions": [{"Name": "AgentName", "Value": "support-agent"}]
        }
    ])

try:
    # CloudWatch accepts max 1000 metrics per call
    cloudwatch.put_metric_data(Namespace=NAMESPACE, MetricData=metric_data[:1000])
    print(f"✅ Published {len(metric_data)} metric data points to CloudWatch")
    print(f"   Namespace: {NAMESPACE}")
    print(f"   Metrics: AgentLatency, TokenUsage, Errors")
except Exception as e:
    print(f"⚠️ CloudWatch publish: {e}")

✅ Published 12 metric data points to CloudWatch
   Namespace: AgentCore/Lab
   Metrics: AgentLatency, TokenUsage, Errors


## 5. Create a CloudWatch Dashboard

In [8]:
DASHBOARD_NAME = "AgentCore-Lab-Dashboard"

dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0, "y": 0, "width": 12, "height": 6,
            "properties": {
                "title": "Agent Latency (ms)",
                "metrics": [[NAMESPACE, "AgentLatency", "AgentName", "support-agent"]],
                "period": 60, "stat": "Average", "region": REGION
            }
        },
        {
            "type": "metric",
            "x": 12, "y": 0, "width": 12, "height": 6,
            "properties": {
                "title": "Token Usage",
                "metrics": [[NAMESPACE, "TokenUsage", "AgentName", "support-agent"]],
                "period": 60, "stat": "Sum", "region": REGION
            }
        },
        {
            "type": "metric",
            "x": 0, "y": 6, "width": 12, "height": 6,
            "properties": {
                "title": "Error Count",
                "metrics": [[NAMESPACE, "Errors", "AgentName", "support-agent"]],
                "period": 60, "stat": "Sum", "region": REGION
            }
        }
    ]
}

try:
    cloudwatch.put_dashboard(
        DashboardName=DASHBOARD_NAME,
        DashboardBody=json.dumps(dashboard_body)
    )
    print(f"✅ Dashboard created: {DASHBOARD_NAME}")
    print(f"   View at: https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#dashboards:name={DASHBOARD_NAME}")
except Exception as e:
    print(f"⚠️ Dashboard creation: {e}")

✅ Dashboard created: AgentCore-Lab-Dashboard
   View at: https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#dashboards:name=AgentCore-Lab-Dashboard


## 6. Debugging a Failed Tool Call

When an agent fails, traces help you pinpoint exactly where and why.

In [9]:
# Trigger a failure and examine the trace
print("Triggering a tool failure for debugging demo...\n")

start = time.time()
try:
    result = agent("Please use the failing_tool with input 'trigger error please'")
    print(f"Agent response: {result.message}")
except Exception as e:
    print(f"Agent handled the error gracefully")

latency = time.time() - start

# Simulated trace output (in production, this comes from OpenTelemetry)
print(f"\n=== Debug Trace ===")
print(f"Session: debug-session")
print(f"Total latency: {latency:.2f}s")
print(f"Trace:")
print(f"  1. [Agent] Received user message")
print(f"  2. [Model] Reasoning: user wants to use failing_tool")
print(f"  3. [Tool Call] failing_tool(input_text='trigger error please')")
print(f"  4. [Tool Error] Exception: Simulated tool failure: connection timeout")
print(f"  5. [Model] Reasoning: tool failed, inform user")
print(f"  6. [Agent] Response generated")
print(f"\n💡 Root cause: Tool 'failing_tool' raised an exception on input containing 'error'")

Triggering a tool failure for debugging demo...

<thinking> The user has explicitly asked to use the `failing_tool` with the input 'trigger error please'. I will proceed with this request as instructed. </thinking> 
Tool #8: failing_tool
<thinking> The `failing_tool` has produced an error as expected. I will inform the user about the error that occurred. </thinking>

The `failing_tool` has produced the following error: "Error: Exception - Simulated tool failure: connection timeout". This is a simulated failure for testing purposes.Agent response: {'role': 'assistant', 'content': [{'text': '<thinking> The `failing_tool` has produced an error as expected. I will inform the user about the error that occurred. </thinking>\n\nThe `failing_tool` has produced the following error: "Error: Exception - Simulated tool failure: connection timeout". This is a simulated failure for testing purposes.'}]}

=== Debug Trace ===
Session: debug-session
Total latency: 2.01s
Trace:
  1. [Agent] Received use

## Key Observability Metrics for Agents

| Metric | What It Tells You | Alert Threshold |
|--------|-------------------|----------------|
| **Latency (p95)** | User experience degradation | > 10s |
| **Token usage** | Cost tracking and anomaly detection | > 2x baseline |
| **Error rate** | Agent reliability | > 5% |
| **Tool call failures** | Integration health | > 10% per tool |
| **Session duration** | Task complexity trends | > 5 minutes |
| **Goal success rate** | Agent effectiveness | < 80% |

## 🧹 Cleanup (Optional)

Uncomment to delete CloudWatch resources.

In [10]:
# # Cleanup
# try:
#     cloudwatch.delete_dashboards(DashboardNames=[DASHBOARD_NAME])
#     print(f"✅ Dashboard '{DASHBOARD_NAME}' deleted")
# except Exception as e:
#     print(f"Cleanup: {e}")